# RNN Text Generation with PyTorch (Vanilla RNN, LSTM, GRU)

---

# Step 1: Import Libraries

In [23]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

In [24]:
print("Libraries loaded.")
print("PyTorch version:", torch.__version__)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

Libraries loaded.
PyTorch version: 2.7.0
Device: cpu


---

# Step 2: Load Dataset

In [25]:
df = pd.read_csv("./datasets/imdb_balanced_10k.csv")

In [26]:
print("Dataset size:", len(df))

print("Columns:", df.columns.tolist())

print("\nFirst review preview:")
print(df["text"].iloc[0][:500])

print("\nFirst label:")
print(df["label"].iloc[0])

Dataset size: 10000
Columns: ['text', 'label']

First review preview:
Unreal "movie", what were these people on?? A mix of French Upstairs Downstairs, mating horses,porn (not suggested, its pretty full on for a film) & bestiality with a bit of Benny Hill music & chase scenes thrown in, its sounds crazy & its even more so to watch. **spoiler** It plods along in a tedious fashion for quite a while,.... then a Lamb does a runner, prompting woman in period dress to run off after it, she goes into the woods where she is set upon by an erect "penis" attached to a man in

First label:
0


---

# Step 3: Build Corpus

We keep part of the dataset for faster training.

In [27]:
MAX_REVIEWS = 2000

texts = df["text"].astype(str).iloc[:MAX_REVIEWS].tolist()

corpus = " ".join(texts).lower()

In [28]:
print("Reviews used:", len(texts))

print("Corpus length:", len(corpus))

print("\nFirst 500 characters:")
print(corpus[:500])

Reviews used: 2000
Corpus length: 2669211

First 500 characters:
unreal "movie", what were these people on?? a mix of french upstairs downstairs, mating horses,porn (not suggested, its pretty full on for a film) & bestiality with a bit of benny hill music & chase scenes thrown in, its sounds crazy & its even more so to watch. **spoiler** it plods along in a tedious fashion for quite a while,.... then a lamb does a runner, prompting woman in period dress to run off after it, she goes into the woods where she is set upon by an erect "penis" attached to a man in


---

# Step 4: Build Character Vocabulary

In [29]:
chars = sorted(list(set(corpus)))

char_to_idx = {
    ch: idx
    for idx, ch in enumerate(chars)
}

idx_to_char = {
    idx: ch
    for ch, idx in char_to_idx.items()
}

In [30]:
print("Unique characters:", len(chars))

print("\nFirst 30 characters:")
print(chars[:30])

Unique characters: 103

First 30 characters:
['\t', ' ', '!', '"', '#', '$', '%', '&', "'", '(', ')', '*', '+', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '<']


---

# Step 5: Encode Corpus

In [31]:
encoded = np.array([
    char_to_idx[ch]
    for ch in corpus
], dtype=np.int64)

In [32]:
print("Encoded length:", len(encoded))

print("First 100 values:")
print(encoded[:100])

Encoded length: 2669211
First 100 values:
[59 52 56 43 39 50  1  3 51 53 60 47 43  3 13  1 61 46 39 58  1 61 43 56
 43  1 58 46 43 57 43  1 54 43 53 54 50 43  1 53 52 32 32  1 39  1 51 47
 62  1 53 44  1 44 56 43 52 41 46  1 59 54 57 58 39 47 56 57  1 42 53 61
 52 57 58 39 47 56 57 13  1 51 39 58 47 52 45  1 46 53 56 57 43 57 13 54
 53 56 52  1]


---

# Step 6: Build Sequences

In [33]:
SEQ_LENGTH = 80

X = []
y = []

for i in range(len(encoded) - SEQ_LENGTH):
    X.append(encoded[i:i+SEQ_LENGTH])
    y.append(encoded[i+SEQ_LENGTH])

X = np.array(X, dtype=np.int64)

y = np.array(y, dtype=np.int64)

In [34]:
print("X shape:", X.shape)

print("y shape:", y.shape)

print("\nFirst target character:")
print(idx_to_char[y[0]])

X shape: (2669131, 80)
y shape: (2669131,)

First target character:
 


---

# Step 7: Reduce Sample Count

In [35]:
MAX_SAMPLES = 40000

X = X[:MAX_SAMPLES]

y = y[:MAX_SAMPLES]

In [36]:
print("Samples used:", len(X))

Samples used: 40000


---

# Step 8: Build Dataset Class

In [37]:
class TextDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [38]:
dataset = TextDataset(X, y)

loader = DataLoader(
    dataset,
    batch_size=128,
    shuffle=True
)

print("Batches per epoch:", len(loader))

Batches per epoch: 313


---

# Step 9: Build Generic RNN Model

In [39]:
class CharRNN(nn.Module):
    def __init__(
        self,
        vocab_size,
        embed_dim,
        hidden_dim,
        cell_type="rnn"
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embed_dim
        )

        if cell_type == "rnn":
            self.rnn = nn.RNN(
                embed_dim,
                hidden_dim,
                batch_first=True
            )

        elif cell_type == "lstm":
            self.rnn = nn.LSTM(
                embed_dim,
                hidden_dim,
                batch_first=True
            )

        elif cell_type == "gru":
            self.rnn = nn.GRU(
                embed_dim,
                hidden_dim,
                batch_first=True
            )

        self.fc = nn.Linear(
            hidden_dim,
            vocab_size
        )

        self.cell_type = cell_type

    def forward(self, x):
        x = self.embedding(x)

        output, hidden = self.rnn(x)

        output = output[:, -1, :]

        logits = self.fc(output)

        return logits

In [40]:
print("Model class ready.")

Model class ready.


---

# Step 10: Build Training Function

In [41]:
def train_model(model, loader, epochs=20):
    model.to(device)

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=0.001
    )

    for epoch in range(epochs):
        total_loss = 0

        for batch_X, batch_y in loader:
            batch_X = batch_X.to(device)

            batch_y = batch_y.to(device)

            optimizer.zero_grad()

            logits = model(batch_X)

            loss = criterion(
                logits,
                batch_y
            )

            loss.backward()

            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(loader)

        print(
            f"Epoch {epoch+1}/{epochs} Loss: {avg_loss:.4f}"
        )

    return model

In [42]:
print("Training function ready.")

Training function ready.


---

# Step 11: Train Vanilla RNN

In [43]:
vanilla_model = CharRNN(
    vocab_size=len(chars),
    embed_dim=64,
    hidden_dim=128,
    cell_type="rnn"
)

In [44]:
vanilla_model = train_model(
    vanilla_model,
    loader
    )

Epoch 1/20 Loss: 2.5835
Epoch 2/20 Loss: 2.1769
Epoch 3/20 Loss: 2.0437
Epoch 4/20 Loss: 1.9515
Epoch 5/20 Loss: 1.8860
Epoch 6/20 Loss: 1.8328
Epoch 7/20 Loss: 1.7860
Epoch 8/20 Loss: 1.7487
Epoch 9/20 Loss: 1.7174
Epoch 10/20 Loss: 1.6875
Epoch 11/20 Loss: 1.6625
Epoch 12/20 Loss: 1.6367
Epoch 13/20 Loss: 1.6148
Epoch 14/20 Loss: 1.5930
Epoch 15/20 Loss: 1.5730
Epoch 16/20 Loss: 1.5545
Epoch 17/20 Loss: 1.5378
Epoch 18/20 Loss: 1.5203
Epoch 19/20 Loss: 1.5028
Epoch 20/20 Loss: 1.4895


---

# Step 12: Train LSTM

In [45]:
lstm_model = CharRNN(
    vocab_size=len(chars),
    embed_dim=64,
    hidden_dim=128,
    cell_type="lstm"
)

In [46]:
lstm_model = train_model(
    lstm_model,
    loader
)

Epoch 1/20 Loss: 2.7205
Epoch 2/20 Loss: 2.2860
Epoch 3/20 Loss: 2.1554
Epoch 4/20 Loss: 2.0715
Epoch 5/20 Loss: 2.0022
Epoch 6/20 Loss: 1.9466
Epoch 7/20 Loss: 1.9000
Epoch 8/20 Loss: 1.8567
Epoch 9/20 Loss: 1.8208
Epoch 10/20 Loss: 1.7868
Epoch 11/20 Loss: 1.7550
Epoch 12/20 Loss: 1.7262
Epoch 13/20 Loss: 1.6988
Epoch 14/20 Loss: 1.6726
Epoch 15/20 Loss: 1.6475
Epoch 16/20 Loss: 1.6230
Epoch 17/20 Loss: 1.6007
Epoch 18/20 Loss: 1.5781
Epoch 19/20 Loss: 1.5558
Epoch 20/20 Loss: 1.5346


---

# Step 13: Train GRU

In [47]:
gru_model = CharRNN(
    vocab_size=len(chars),
    embed_dim=64,
    hidden_dim=128,
    cell_type="gru"
)

In [48]:
gru_model = train_model(
    gru_model,
    loader
)

Epoch 1/20 Loss: 2.5946
Epoch 2/20 Loss: 2.1436
Epoch 3/20 Loss: 1.9890
Epoch 4/20 Loss: 1.8863
Epoch 5/20 Loss: 1.8111
Epoch 6/20 Loss: 1.7491
Epoch 7/20 Loss: 1.6953
Epoch 8/20 Loss: 1.6491
Epoch 9/20 Loss: 1.6029
Epoch 10/20 Loss: 1.5631
Epoch 11/20 Loss: 1.5258
Epoch 12/20 Loss: 1.4931
Epoch 13/20 Loss: 1.4586
Epoch 14/20 Loss: 1.4288
Epoch 15/20 Loss: 1.4007
Epoch 16/20 Loss: 1.3725
Epoch 17/20 Loss: 1.3459
Epoch 18/20 Loss: 1.3208
Epoch 19/20 Loss: 1.2964
Epoch 20/20 Loss: 1.2726


---

# Step 14: Build Text Generator

In [49]:
def generate_text(
    model,
    seed_text,
    length=400
):
    model.eval()

    generated = seed_text.lower()

    for _ in range(length):
        sequence = [
            char_to_idx.get(ch, 0)
            for ch in generated[-SEQ_LENGTH:]
        ]

        if len(sequence) < SEQ_LENGTH:
            sequence = [0] * (SEQ_LENGTH - len(sequence)) + sequence

        x = torch.tensor(
            [sequence],
            dtype=torch.long
        ).to(device)

        with torch.no_grad():
            logits = model(x)

        next_idx = torch.argmax(
            logits,
            dim=1
        ).item()

        next_char = idx_to_char[next_idx]

        generated += next_char

    return generated

In [50]:
print("Generation function ready.")

Generation function ready.


---

# Step 15: Generate Text with Vanilla RNN

In [51]:
seed = "this movie"

In [52]:
vanilla_output = generate_text(
    vanilla_model,
    seed,
    length=500
)

In [53]:
print("Vanilla RNN Output:\n")
print(vanilla_output)

Vanilla RNN Output:

this movie and the script and gadget the series to the series to the series to the series to the series to the series to the series to the series to the series to the series to the series to the series to the series to the series to the series to the series to the series to the series to the series to the series to the series to the series to the series to the series to the series to the series to the series to the series to the series to the series to the series to the series to the series to the series 


---

# Step 16: Generate Text with LSTM

In [54]:
lstm_output = generate_text(
    lstm_model,
    seed,
    length=500
)

In [55]:
print("LSTM Output:\n")
print(lstm_output)

LSTM Output:

this movie and the seen the seen to the seen to the seen to the seen to the seen to the seen to the seen to the seen to the seen to the seen to the seen to the seen to the seen to the seen to the seen to the seen to the seen to the seen to the seen to the seen to the seen to the seen to the seen to the seen to the seen to the seen to the seen to the seen to the seen to the seen to the seen to the seen to the seen to the seen to the seen to the seen to the seen to the seen to the seen to the seen to the se


---

# Step 17: Generate Text with GRU

In [56]:
gru_output = generate_text(
    gru_model,
    seed,
    length=500
)

In [57]:
print("GRU Output:\n")
print(gru_output)

GRU Output:

this movie and the story of the story of the story of the story of the story of the story of the story of the story of the story of the story of the story of the story of the story of the story of the story of the story of the story of the story of the story of the story of the story of the story of the story of the story of the story of the story of the story of the story of the story of the story of the story of the story of the story of the story of the story of the story of the story of the story of t


---

# Step 18: Compare Outputs

In [58]:
print("Seed:", seed)

print("\nVanilla Sample:")
print(vanilla_output[:300])

print("\nLSTM Sample:")
print(lstm_output[:300])

print("\nGRU Sample:")
print(gru_output[:300])

Seed: this movie

Vanilla Sample:
this movie and the script and gadget the series to the series to the series to the series to the series to the series to the series to the series to the series to the series to the series to the series to the series to the series to the series to the series to the series to the series to the series 

LSTM Sample:
this movie and the seen the seen to the seen to the seen to the seen to the seen to the seen to the seen to the seen to the seen to the seen to the seen to the seen to the seen to the seen to the seen to the seen to the seen to the seen to the seen to the seen to the seen to the seen to the seen to 

GRU Sample:
this movie and the story of the story of the story of the story of the story of the story of the story of the story of the story of the story of the story of the story of the story of the story of the story of the story of the story of the story of the story of the story of the story of the story of


---

# Step 19: Try Different Seeds

In [59]:
seeds = [
    "the movie",
    "i loved",
    "this film",
    "one of the"
]

In [60]:
for seed_text in seeds:
    print(f"\nSeed: {seed_text}")
    print("-" * 60)

    sample = generate_text(
        gru_model,
        seed_text,
        length=200
    )

    print(sample)


Seed: the movie
------------------------------------------------------------
the movie and the story of the story of the story of the story of the story of the story of the story of the story of the story of the story of the story of the story of the story of the story of the story of 

Seed: i loved
------------------------------------------------------------
i loved the story of the story of the story of the story of the story of the story of the story of the story of the story of the story of the story of the story of the story of the story of the story of the 

Seed: this film
------------------------------------------------------------
this film is the story of the story of the story of the story of the story of the story of the story of the story of the story of the story of the story of the story of the story of the story of the story of t

Seed: one of the
------------------------------------------------------------
one of the story of the story of the story of the story of th